# SustAInable — Notebook 06: Agency-Facing Summary Report
**Project:** SustAInable — Neighborhood Heat Illness Risk Prediction by ZIP Code  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)

---

## What This Notebook Does

Notebooks 01–05 are for data scientists and technical reviewers. This notebook is for everyone else.

It produces a plain-language summary of SustAInable's findings — written for city health department leadership, emergency management coordinators, climate equity officers, and elected officials who need to understand what the model found and what to do about it, without needing to understand gradient boosting.

**Outputs:**
- Executive summary with key metrics in plain language
- Top 20 highest-risk ZIP codes in briefing-ready format
- Risk tier breakdown by region and urban type
- `sustainable_agency_report.csv` — the operational deliverable
- `sustainable_executive_summary.md` — ready to paste into any report, grant, or proposal

In [ ]:
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import fbeta_score, roc_auc_score, recall_score, precision_score
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style='whitegrid')
THRESHOLD = 0.30
REPORT_DATE = datetime.today().strftime('%B %d, %Y')
print(f'Report date: {REPORT_DATE}')

## 1. Load Everything & Score Full Dataset

In [ ]:
model = joblib.load('sustainable_xgboost_model.pkl')
X_holdout = np.load('sustainable_X_holdout.npy')
y_holdout = np.load('sustainable_y_holdout.npy')
feature_cols = joblib.load('sustainable_feature_cols.pkl')
encoders = joblib.load('sustainable_encoders.pkl')
df_orig = pd.read_csv('sustainable_zip_data.csv')

# Reconstruct features for full dataset scoring
df_s = df_orig.copy()
df_s['region_encoded'] = encoders['region'].transform(df_s['region'])
df_s['urban_type_encoded'] = encoders['urban_type'].transform(df_s['urban_type'])
scaler = MinMaxScaler()
vuln = df_s[['poverty_rate_pct','pct_65_plus','uninsured_rate_pct',
              'prev_heart_disease_pct','prev_diabetes_pct']].values
df_s['social_vulnerability_index'] = scaler.fit_transform(vuln).mean(axis=1)
df_s['infrastructure_gap'] = (
    df_s['extreme_heat_days_annual'] / df_s['extreme_heat_days_annual'].max() -
    df_s['ac_access_pct'] / 100 -
    df_s['cooling_centers_per_10k'] / df_s['cooling_centers_per_10k'].max()
)
df_s['heat_x_vulnerability'] = df_s['avg_summer_temp_f'] / 100 * df_s['social_vulnerability_index']
df_s['canopy_deficit'] = df_s['impervious_surface_pct'] - df_s['tree_canopy_coverage_pct']

full_probs = model.predict_proba(df_s[feature_cols].values)[:, 1]

report_df = df_orig[['zip_code','region','urban_type','avg_summer_temp_f',
                      'extreme_heat_days_annual','poverty_rate_pct',
                      'ac_access_pct','pct_65_plus','heat_hazard_index']].copy()
report_df['risk_score'] = full_probs.round(3)
report_df['risk_flag'] = (full_probs >= THRESHOLD).astype(int)
report_df['priority_tier'] = pd.cut(
    full_probs, bins=[0, 0.30, 0.60, 0.80, 1.0],
    labels=['Monitor', 'Elevated', 'High', 'Critical'])
report_df['recommended_action'] = report_df['priority_tier'].map({
    'Monitor':  'Include in seasonal heat planning; monitor conditions',
    'Elevated': 'Schedule proactive outreach within 30 days',
    'High':     'Deploy cooling resources and wellness checks before next heat event',
    'Critical': 'Immediate action — coordinate emergency cooling and health services'
})
report_df = report_df.sort_values('risk_score', ascending=False).reset_index(drop=True)
report_df.to_csv('sustainable_agency_report.csv', index=False)
print(f'Saved: sustainable_agency_report.csv ({len(report_df):,} ZIP codes)')

## 2. Executive Summary Numbers

In [ ]:
y_prob_h = model.predict_proba(X_holdout)[:, 1]
y_pred_h = (y_prob_h >= THRESHOLD).astype(int)

recall = recall_score(y_holdout, y_pred_h)
precision = precision_score(y_holdout, y_pred_h)
f2 = fbeta_score(y_holdout, y_pred_h, beta=2)
auc = roc_auc_score(y_holdout, y_prob_h)

flagged = report_df['risk_flag'].sum()
critical = (report_df['priority_tier'] == 'Critical').sum()
high = (report_df['priority_tier'] == 'High').sum()
elevated = (report_df['priority_tier'] == 'Elevated').sum()

print('=== SustAInable Executive Summary ===')
print(f'  Total ZIP codes scored:              {len(report_df):,}')
print(f'  ZIP codes flagged for intervention:  {int(flagged):,} ({flagged/len(report_df):.1%})')
print(f'  — Critical (immediate action):       {int(critical)}')
print(f'  — High (before next heat event):     {int(high)}')
print(f'  — Elevated (proactive outreach):     {int(elevated)}')
print(f'  Recall (vulnerable ZIPs caught):     {recall:.1%}')
print(f'  Precision (flag accuracy):           {precision:.1%}')
print(f'  F2-Score:                            {f2:.3f}')
print(f'  AUC-ROC:                             {auc:.3f}')

## 3. Top 20 Highest-Risk ZIP Codes

In [ ]:
top20 = report_df.head(20)[[
    'zip_code','region','urban_type','risk_score','priority_tier',
    'recommended_action','avg_summer_temp_f','poverty_rate_pct',
    'ac_access_pct','pct_65_plus'
]].copy()
top20['avg_summer_temp_f'] = top20['avg_summer_temp_f'].round(1)
top20['poverty_rate_pct'] = top20['poverty_rate_pct'].round(1)
top20['ac_access_pct'] = top20['ac_access_pct'].round(1)
top20['pct_65_plus'] = top20['pct_65_plus'].round(1)
top20.index = range(1, 21)
top20.index.name = 'Rank'
print('TOP 20 HIGHEST-RISK ZIP CODES')
print('=' * 80)
print(top20.to_string())

## 4. Risk Tier Breakdown by Region and Urban Type

In [ ]:
tier_order = ['Monitor', 'Elevated', 'High', 'Critical']
colors = ['#81C784', '#FFB74D', '#EF5350', '#B71C1C']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'SustAInable — Risk Tier Breakdown\nReport Date: {REPORT_DATE} | Threshold: {THRESHOLD}',
             fontweight='bold', fontsize=13)

# By region
tier_region = report_df.groupby(['region','priority_tier']).size().unstack(fill_value=0)
tier_region = tier_region.reindex(columns=tier_order, fill_value=0)
tier_region.plot(kind='bar', ax=axes[0], color=colors, alpha=0.9, edgecolor='white', width=0.75)
axes[0].set_title('By Region', fontweight='bold')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('ZIP Codes')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
axes[0].legend(title='Priority Tier')

# By urban type
tier_urban = report_df.groupby(['urban_type','priority_tier']).size().unstack(fill_value=0)
tier_urban = tier_urban.reindex(columns=tier_order, fill_value=0)
tier_urban.plot(kind='bar', ax=axes[1], color=colors, alpha=0.9, edgecolor='white', width=0.65)
axes[1].set_title('By Urban Type', fontweight='bold')
axes[1].set_xlabel('Urban Type')
axes[1].set_ylabel('ZIP Codes')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].legend(title='Priority Tier')

plt.tight_layout()
plt.savefig('sustainable_agency_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_agency_tiers.png')

## 5. Generate Executive Summary Markdown

In [ ]:
md = f"""# SustAInable — Executive Summary
**Report Date:** {REPORT_DATE}  
**Decision Threshold:** {THRESHOLD}  
**Model:** XGBoost Supervised Classification  
**Produced by:** Nico Meyering, MPA — [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)

---

## What SustAInable Found

SustAInable scored **{len(report_df):,} ZIP codes** across {report_df['region'].nunique()} U.S. regions for elevated heat illness risk during extreme heat events.

Of those, **{int(flagged):,} ZIP codes** ({flagged/len(report_df):.1%}) are flagged for proactive public health intervention before the next heat event.

| Priority Tier | ZIP Codes | Recommended Action |
|---|---|---|
| 🔴 Critical | {int(critical)} | Immediate action — coordinate emergency cooling and health services |
| 🟠 High | {int(high)} | Deploy cooling resources and wellness checks before next heat event |
| 🟡 Elevated | {int(elevated)} | Schedule proactive outreach within 30 days |
| 🟢 Monitor | {len(report_df) - int(flagged)} | Include in seasonal planning; monitor conditions |

---

## Model Performance

| Metric | Value | What It Means |
|---|---|---|
| Recall | {recall:.1%} | Of high-risk ZIP codes, SustAInable identified {recall:.1%} before a heat event |
| Precision | {precision:.1%} | Of ZIP codes flagged, {precision:.1%} were genuinely at elevated risk |
| F2-Score | {f2:.3f} | Primary metric — recall weighted twice as heavily as precision |
| AUC-ROC | {auc:.3f} | Overall model discrimination (1.0 = perfect, 0.5 = random) |

---

## The Core Judgment

In 2023, heat killed **2,415 Americans** — the highest count in 45 years. That number is an undercount.  
Missing a high-risk community during a heat wave means preventable deaths.  
Flagging a lower-risk community means unnecessary outreach — wasteful, not catastrophic.

SustAInable is calibrated to minimize false negatives. No community should be left out of the response plan because a model was too conservative.

---

## Next Steps for Agency Partners

1. **Share your jurisdiction's data** — SustAInable retrains on local CDC PLACES and weather records in one session
2. **Select your operating threshold** — full trade-off curve in `sustainable_threshold_sensitivity.png`
3. **Import the agency report** — `sustainable_agency_report.csv` is formatted for direct use in outreach planning

Contact: **nicmeyering@gmail.com** · [LinkedIn](https://www.linkedin.com/in/nicolasmeyering)
"""

with open('sustainable_executive_summary.md', 'w') as f:
    f.write(md)
print('Saved: sustainable_executive_summary.md')
print()
print(md)

## All Outputs From This Notebook

| File | Audience | Description |
|---|---|---|
| `sustainable_agency_report.csv` | Health department staff | Full ranked ZIP list with recommended actions |
| `sustainable_agency_tiers.png` | Agency leadership | Risk breakdown by region and urban type |
| `sustainable_executive_summary.md` | Any stakeholder | Plain-language summary for reports and proposals |

---

**SustAInable is ready for real data.** Replace `sustainable_zip_data.csv` with your jurisdiction's records, rerun notebooks 01–06, and all outputs update automatically.

*Equitech Futures Civic Tech Institute, CTI 2026 · Philadelphia, PA*